In [15]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
import torchvision
from torchvision import transforms
from torch.utils.data import TensorDataset, Subset

from layers.WHT import WHTConv2D

In [16]:
# download format
# turns MNIST images to PyTorch tensors and normalizes between [-1,1] centered at 0
train_transform = transforms.Compose([
    transforms.RandomCrop(28, padding=3),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(5), # can be 10
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

test_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

# download data to computer
raw_train_dataset = torchvision.datasets.FashionMNIST(
    root='./data',
    train=True,
    download=True,
    transform=train_transform
)

raw_test_dataset = torchvision.datasets.FashionMNIST(
    root='./data',
    train=False,            # testing data so labels are unknown during training
    download=True,
    transform=test_transform
)

In [17]:
raw_train_subset = Subset(raw_train_dataset, range(20000))
raw_test_subset = Subset(raw_test_dataset, range(2000))

# use raw_train_small and raw_test_small to decrease number of training and test images
# use raw_train_dataset and raw_test_dataset for entire dataset
train_dataset = raw_train_dataset
test_dataset  = raw_test_dataset

In [18]:
# loaders
train_loader = DataLoader(
    train_dataset,
    batch_size=128,         # each epoch is 128 samples
    shuffle=True            # randomize after each training epoch
)

test_loader = DataLoader(
    test_dataset,
    batch_size=256,         # each epoch is 256 samples
    shuffle=False
)

# test shapes of pytorch datasets
images, labels = next(iter(train_loader))
print(images.shape)
print(labels.shape)

# print("train size", len(train_loader.dataset))
# print("test size", len(test_loader.dataset))

torch.Size([128, 1, 28, 28])
torch.Size([128])


In [19]:
# CNN model
class CNN(nn.Module):

    def __init__(self):
        super().__init__()

        #self.wht = WHTConv2D(height=14, width=14, in_channels=32, out_channels=32, pods=1, residual=True)

        self.conv1a = nn.Conv2d(1, 32, 3, padding=1)
        self.conv1b = nn.Conv2d(32, 32, 3, padding=1)

        self.conv2a = nn.Conv2d(32, 64, 3, padding=1)
        self.conv2b = nn.Conv2d(64, 64, 3, padding=1)

        self.conv3a = nn.Conv2d(64, 128, 3, padding=1)
        self.conv3b = nn.Conv2d(128, 128, 3, padding=1)

        self.conv4a = nn.Conv2d(128, 256, 3, padding=1)
        self.conv4b = nn.Conv2d(256, 256, 3, padding=1)

        # a and b batch norms for their corresponding conv layer because batchnorm learns
        self.bn1a = nn.BatchNorm2d(32)
        self.bn2a = nn.BatchNorm2d(64)
        self.bn3a = nn.BatchNorm2d(128)
        self.bn4a = nn.BatchNorm2d(256)

        self.bn1b = nn.BatchNorm2d(32)
        self.bn2b = nn.BatchNorm2d(64)
        self.bn3b = nn.BatchNorm2d(128)
        self.bn4b = nn.BatchNorm2d(256)

        self.pool = nn.MaxPool2d(2, 2)

        #self.drop = nn.Dropout(0.0)

        self.gap = nn.AdaptiveAvgPool2d(1)

        self.fc1 = nn.Linear(256, 10)
        #self.fc2 = nn.Linear(128, 10)
        #self.fc3 = nn.Linear(128, 10)

    def forward(self, x):
        x = F.relu(self.bn1a(self.conv1a(x)))
        x = self.pool(F.relu(self.bn1b(self.conv1b(x))))

        x = F.relu(self.bn2a(self.conv2a(x)))
        x = self.pool(F.relu(self.bn2b(self.conv2b(x))))

        x = F.relu(self.bn3a(self.conv3a(x)))
        x = self.pool(F.relu(self.bn3b(self.conv3b(x))))

        x = F.relu(self.bn4a(self.conv4a(x)))
        x = F.relu(self.bn4b(self.conv4b(x)))

        #x = torch.flatten(x, start_dim=1)

        x = self.gap(x).squeeze(-1).squeeze(-1)
        x = self.fc1(x)
        #x = self.drop(x)
        #x = self.fc2(x)

        #x = self.drop(x)
        #x = self.fc3(x)

        return x

# create model, loss, and optimizer
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

num_epochs = 75 # NUM EPOCHS

model = CNN().to(device)
criterion = nn.CrossEntropyLoss(label_smoothing=0.02)
optimizer = torch.optim.AdamW(model.parameters(), lr=0.001, weight_decay=0.0005)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=num_epochs)

In [20]:
# eval accuracy within training loop
def eval_acc():
    model.eval()
    correct = 0
    total = 0
    
    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)
            pred = model(images).argmax(dim=1)
            total += labels.size(0)
            correct += (pred == labels).sum().item()
    
    model.train()
    return 100 * correct / total

In [21]:
# training loop
for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0

    for images, labels in train_loader:
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    scheduler.step()
    print(f"Test Accuracy: {eval_acc():.2f}%")
    print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {running_loss/len(train_loader):.4f}")

Test Accuracy: 86.54%
Epoch [1/75], Loss: 0.5883
Test Accuracy: 87.11%
Epoch [2/75], Loss: 0.4413
Test Accuracy: 90.70%
Epoch [3/75], Loss: 0.4019
Test Accuracy: 90.93%
Epoch [4/75], Loss: 0.3788
Test Accuracy: 90.25%
Epoch [5/75], Loss: 0.3641
Test Accuracy: 91.13%
Epoch [6/75], Loss: 0.3526
Test Accuracy: 92.02%
Epoch [7/75], Loss: 0.3406
Test Accuracy: 92.40%
Epoch [8/75], Loss: 0.3321
Test Accuracy: 92.27%
Epoch [9/75], Loss: 0.3278
Test Accuracy: 91.87%
Epoch [10/75], Loss: 0.3172
Test Accuracy: 92.29%
Epoch [11/75], Loss: 0.3126
Test Accuracy: 92.75%
Epoch [12/75], Loss: 0.3072
Test Accuracy: 92.92%
Epoch [13/75], Loss: 0.3030
Test Accuracy: 92.98%
Epoch [14/75], Loss: 0.2961
Test Accuracy: 93.44%
Epoch [15/75], Loss: 0.2948
Test Accuracy: 93.64%
Epoch [16/75], Loss: 0.2883
Test Accuracy: 93.09%
Epoch [17/75], Loss: 0.2841
Test Accuracy: 93.37%
Epoch [18/75], Loss: 0.2802
Test Accuracy: 93.84%
Epoch [19/75], Loss: 0.2783
Test Accuracy: 93.10%
Epoch [20/75], Loss: 0.2738
Test Accu

In [22]:
# get accuracy
model.eval()
correct = 0
total = 0

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)
        _, predicted = torch.max(outputs, 1)

        total += labels.size(0)
        correct += (predicted == labels).sum().item()

print(f"Test Accuracy: {100 * correct / total:.2f}%")

Test Accuracy: 95.13%
